# Data Cleaning and Timestamp Processing

Produces `data/processed/demand_clean.csv` — the file the feature-engineering notebook, model-training notebooks, and the local backend API (`backend/main.py`) all read from. Source: `data/raw/load_forecasting_dataset_corrected.csv` (see `01_data_understanding.ipynb` for the data-authenticity note — this is treated as a synthetic Sri Lanka-style dataset, not validated real CEB telemetry). Column names are standardized here (`timestamp`, `demand`, plus weather and passthrough columns) so every downstream stage can rely on them via `config.yaml`. Native 15-minute readings are resampled to hourly to match the project's hourly design (SARIMA seasonal period 24, week-long lags, etc.).

In [ ]:
import os
import sys
from pathlib import Path

import yaml

# Works both in the local repo and on Kaggle. Locally, config.yaml is read
# from disk. On Kaggle there is no repo checkout -- only whatever cells you
# paste -- so the config is built inline instead, pointed at the attached
# dataset under /kaggle/input. IS_KAGGLE is detected via KAGGLE_KERNEL_RUN_TYPE
# (always set on Kaggle) rather than checking /kaggle/input directly, since
# that directory doesn't exist until a dataset is attached -- checking it
# directly would silently fall through to the local branch and fail with a
# confusing "config.yaml not found" instead. The dataset is then located by
# searching /kaggle/input for the raw CSV by name rather than assuming a
# fixed mount depth, since Kaggle has been observed to mount an attached
# dataset at different depths depending on how it's attached (flat
# /kaggle/input/<slug>/... vs nested /kaggle/input/datasets/<owner>/<slug>/...).
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    RAW_FILENAME = "load_forecasting_dataset_corrected.csv"
    matches = list(Path("/kaggle/input").rglob(RAW_FILENAME))
    if not matches:
        raise RuntimeError(
            f"Could not find {RAW_FILENAME} anywhere under /kaggle/input -- "
            "attach the dataset containing it via the notebook's Data panel."
        )
    if len(matches) > 1:
        raise RuntimeError(
            f"Found {len(matches)} copies of {RAW_FILENAME} under /kaggle/input: "
            f"{matches} -- remove the extras or point RAW_DIR at the right one manually."
        )
    print("Found raw data at:", matches[0])

    REPO_ROOT = Path("/kaggle/working")
    config = {
        "data": {
            "raw_path": str(matches[0].parent),
            "processed_path": "/kaggle/working/data/processed",
            "processed_file": "/kaggle/working/data/processed/demand_clean.csv",
            "features_file": "/kaggle/working/data/processed/features.csv",
            "timestamp_col": "timestamp",
            "target_col": "demand",
            "frequency": "h",
        },
        "peak_demand": {"percentile_threshold": 0.95, "seasonal": True},
    }
else:
    def find_repo_root(start: Path) -> Path:
        for parent in [start, *start.parents]:
            if (parent / "config.yaml").exists():
                return parent
        raise FileNotFoundError("config.yaml not found in any parent directory")

    REPO_ROOT = find_repo_root(Path.cwd())
    sys.path.insert(0, str(REPO_ROOT))

    with open(REPO_ROOT / "config.yaml") as f:
        config = yaml.safe_load(f)

config

In [11]:
import numpy as np
import pandas as pd

# On Kaggle, src/ isn't on disk (only pasted cells are) -- these are
# identical inline copies of src/data/load_data.py and src/data/clean_data.py.
# Keep them in sync with src/ if that logic ever changes.
if IS_KAGGLE:
    def set_timestamp_index(df, timestamp_col="timestamp"):
        df = df.set_index(timestamp_col)
        return df.sort_index()

    def detect_missing_intervals(df, freq="h"):
        full_range = pd.date_range(df.index.min(), df.index.max(), freq=freq)
        return full_range.difference(df.index)

    def remove_duplicate_timestamps(df):
        return df[~df.index.duplicated(keep="first")]

    def fill_missing_values(df, method="time"):
        return df.interpolate(method=method, limit_direction="forward")

    def flag_invalid_demand(df, target_col):
        df = df.copy()
        df["invalid_demand"] = df[target_col] <= 0
        return df
else:
    from src.data.clean_data import fill_missing_values, flag_invalid_demand, remove_duplicate_timestamps
    from src.data.load_data import detect_missing_intervals, set_timestamp_index

RAW_DIR = REPO_ROOT / config["data"]["raw_path"]
RAW_FILE = "load_forecasting_dataset_corrected.csv"
PROCESSED_PATH = REPO_ROOT / config["data"]["processed_file"]
TIMESTAMP_COL = config["data"]["timestamp_col"]
TARGET_COL = config["data"]["target_col"]
FREQ = config["data"]["frequency"]

## Load and standardize columns

Renames the raw CSV's columns to the standardized names the rest of the pipeline expects. Economic columns (GDP, electricity price, per-capita energy use) and the public-event flag are carried through as passthrough columns — not yet used in feature engineering, but preserved for future extension.

In [12]:
raw_df = pd.read_csv(RAW_DIR )

raw_df[TIMESTAMP_COL] = pd.to_datetime(raw_df["Timestamp"], format="%m/%d/%Y %H:%M")

RENAME_MAP = {
    "Load Demand (kW)": TARGET_COL,
    "Temperature (°C)": "temperature",
    "Humidity (%)": "humidity",
    "Wind Speed (m/s)": "wind_speed",
    "Rainfall (mm)": "rainfall",
    "Solar Irradiance (W/m²)": "solar_irradiance",
    "GDP (LKR)": "gdp",
    "Per Capita Energy Use (kWh)": "per_capita_energy_use",
    "Electricity Price (LKR/kWh)": "electricity_price",
    "Public Event": "public_event",
}
raw_df = raw_df.rename(columns=RENAME_MAP)

KEEP_COLS = [TIMESTAMP_COL] + list(RENAME_MAP.values())
raw_df = raw_df[KEEP_COLS]
raw_df.head()

,timestamp,demand,temperature,humidity,wind_speed,rainfall,solar_irradiance,gdp,per_capita_energy_use,electricity_price,public_event
0,2020-01-01 00:00:00,1599.342831,28.993428,75.011269,1.053861,4.140513,185.892561,925.621430,502.915605,20.454440,0
1,2020-01-01 00:15:00,1472.347140,27.723471,77.024015,1.085152,9.446997,281.782650,1020.823521,497.286366,27.776449,0
2,2020-01-01 00:30:00,1629.537708,29.295377,74.732958,3.363800,4.265813,328.942058,1028.847455,488.816292,21.097420,0
3,2020-01-01 00:45:00,1804.605971,31.046060,87.615995,2.539148,1.038103,336.407064,937.963002,468.038834,26.032137,1
4,2020-01-01 01:00:00,1453.169325,27.531693,79.709858,1.366819,4.201393,205.494256,934.477462,488.565716,27.079114,0


## Sort chronologically and set the timestamp index

In [13]:
df = set_timestamp_index(raw_df, TIMESTAMP_COL)
df.head()

,demand,temperature,humidity,wind_speed,rainfall,solar_irradiance,gdp,per_capita_energy_use,electricity_price,public_event
timestamp,,,,,,,,,,
2020-01-01 00:00:00,1599.342831,28.993428,75.011269,1.053861,4.140513,185.892561,925.621430,502.915605,20.454440,0
2020-01-01 00:15:00,1472.347140,27.723471,77.024015,1.085152,9.446997,281.782650,1020.823521,497.286366,27.776449,0
2020-01-01 00:30:00,1629.537708,29.295377,74.732958,3.363800,4.265813,328.942058,1028.847455,488.816292,21.097420,0
2020-01-01 00:45:00,1804.605971,31.046060,87.615995,2.539148,1.038103,336.407064,937.963002,468.038834,26.032137,1
2020-01-01 01:00:00,1453.169325,27.531693,79.709858,1.366819,4.201393,205.494256,934.477462,488.565716,27.079114,0


## Remove duplicate timestamps

In [14]:
before = len(df)
df = remove_duplicate_timestamps(df)
print(f"Removed {before - len(df)} duplicate timestamp rows")

Removed 0 duplicate timestamp rows


## Resample from 15-minute to hourly

The raw data is recorded every 15 minutes; the rest of this pipeline (lag windows, SARIMA's 24-step seasonal period, LSTM sequence length) is designed around hourly steps. Demand and weather columns are averaged; `public_event` uses the max so an event anywhere in the hour marks the whole hour; the economic passthrough columns are averaged too (they change slowly, so this is a no-op in practice).

In [15]:
AGG_MAX_COLS = ["public_event"]
agg_map = {col: ("max" if col in AGG_MAX_COLS else "mean") for col in df.columns}

df = df.resample(FREQ).agg(agg_map)
df.index.name = TIMESTAMP_COL
print(f"Resampled to '{FREQ}': {len(df)} hourly rows")
df.head()

Resampled to 'h': 47472 hourly rows


,demand,temperature,humidity,wind_speed,rainfall,solar_irradiance,gdp,per_capita_energy_use,electricity_price,public_event
timestamp,,,,,,,,,,
2020-01-01 00:00:00,1626.458413,29.264584,78.596059,2.010490,4.722857,283.256083,978.313852,489.264274,23.840111,1
2020-01-01 01:00:00,1593.917861,28.939179,78.981324,2.017491,3.804590,229.819735,935.434256,505.759101,24.308850,0
2020-01-01 02:00:00,1457.196911,27.571969,77.364594,1.972281,5.686889,263.726759,965.089335,491.523802,30.762810,0
2020-01-01 03:00:00,1302.073833,26.020738,79.678678,1.615092,5.024458,250.467925,988.898933,495.765616,26.104404,1
2020-01-01 04:00:00,1349.054422,26.490544,80.105671,2.280944,2.721659,255.340872,1032.194114,497.929332,22.300232,0


## Detect and fill missing time intervals

In [16]:
missing_intervals = detect_missing_intervals(df, freq=FREQ)
full_range = pd.date_range(df.index.min(), df.index.max(), freq=FREQ)
print(f"{len(missing_intervals)} missing hourly timestamps out of an expected {len(full_range)}")

df = df.reindex(full_range)
df.index.name = TIMESTAMP_COL

0 missing hourly timestamps out of an expected 47472


## Flag and clear invalid demand values

In [17]:
df = flag_invalid_demand(df, TARGET_COL)
print(f"{df['invalid_demand'].sum()} invalid (<= 0) demand readings flagged")

df.loc[df["invalid_demand"], TARGET_COL] = np.nan
df = df.drop(columns="invalid_demand")

0 invalid (<= 0) demand readings flagged


## Interpolate missing values

Applied across all numeric columns (demand plus weather/passthrough), not just demand, since resampling/reindexing can introduce gaps in any of them.

In [18]:
before_na = df.isna().sum().sum()
df = fill_missing_values(df, method="time")
after_na = df.isna().sum().sum()
print(f"Missing values before interpolation: {before_na}")
print(f"Missing values after interpolation: {after_na}")

Missing values before interpolation: 0
Missing values after interpolation: 0


## Save the cleaned dataset

In [19]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.reset_index().to_csv(PROCESSED_PATH, index=False)
print(f"Saved {len(df)} rows to {PROCESSED_PATH}")

Saved 47472 rows to /kaggle/working/data/processed/demand_clean.csv
